# Alpha Cinema: расчёт выборки для A/B-дизайна

Приложение к [дизайну эксперимента](../docs/AB_TEST_DESIGN.md), подготовлено 5 ноября 2026 года.

Это расчёт планирования будущего эксперимента. Данных проведённого A/B-теста здесь нет. Базовая конверсия 5% — учебное допущение, а не заполняемость кинотеатра или измеренная конверсия сайта.

Код использует только стандартную библиотеку Python 3.8+. Откройте ноутбук в Jupyter или Google Colab и выполните все ячейки сверху вниз. В первой расчётной ячейке можно изменить конверсии, уровень значимости и мощность.


## Метод

Нормальное приближение для двух независимых долей, равные группы, двусторонняя проверка:

`n = ceil((z_(1-alpha/2) * sqrt(2*p_bar*(1-p_bar)) + z_power * sqrt(pA*(1-pA)+pB*(1-pB)))**2 / (pB-pA)**2)`.

`p_bar=(pA+pB)/2`. Округление всегда вверх. Это приближение для планирования, без поправки на непрерывность; для очень редких событий нужен иной расчёт.

Параметры мощности и соотношения групп описаны в [официальной документации statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.NormalIndPower.solve_power.html). Здесь формула рассчитана напрямую, без зависимости от statsmodels и без смешения с формулой через arcsin-эффект Коэна.


In [1]:
from math import ceil, sqrt
from statistics import NormalDist

def sample_per_arm(p_control, p_test, alpha=0.05, power=0.80):
    if not (0 < p_control < 1 and 0 < p_test < 1):
        raise ValueError("Конверсии должны быть между 0 и 1")
    if p_control == p_test:
        raise ValueError("Для расчёта нужен ненулевой эффект")
    if not (0 < alpha < 1 and 0.5 < power < 1):
        raise ValueError("Проверьте alpha и power")
    normal = NormalDist()
    z_alpha = normal.inv_cdf(1 - alpha / 2)
    z_power = normal.inv_cdf(power)
    p_bar = (p_control + p_test) / 2
    numerator = (z_alpha * sqrt(2 * p_bar * (1 - p_bar))
                 + z_power * sqrt(p_control * (1 - p_control)
                                  + p_test * (1 - p_test))) ** 2
    return ceil(numerator / (p_test - p_control) ** 2)

p_control = 0.05  # учебное допущение
p_test = 0.06     # обнаруживаемый рост до 6%
alpha = 0.05
power = 0.80
n_arm = sample_per_arm(p_control, p_test, alpha, power)
operational_arm = ceil(n_arm / 100) * 100
print(f"MDE: {(p_test-p_control)*100:.1f} п.п.")
print(f"Относительное изменение: {(p_test/p_control-1)*100:.0f}%")
print(f"Расчёт на группу: {n_arm:,}; всего: {2*n_arm:,}")
print(f"Ориентир после округления: {operational_arm:,} на группу")


MDE: 1.0 п.п.
Относительное изменение: 20%
Расчёт на группу: 8,158; всего: 16,316
Ориентир после округления: 8,200 на группу


In [2]:
print("Учебные сценарии при базе 5%, alpha=0.05, power=0.80:")
for p_candidate in (0.055, 0.06, 0.065):
    required = sample_per_arm(0.05, p_candidate)
    print(f"5% -> {p_candidate*100:g}%: {required:,} пользователей на группу")

# Внешние контрольные ориентиры для формулы, указанной в документе.
assert sample_per_arm(0.05, 0.06) == 8158
assert sample_per_arm(0.05, 0.055) > sample_per_arm(0.05, 0.06)
assert sample_per_arm(0.05, 0.06, power=0.90) > sample_per_arm(0.05, 0.06)


Учебные сценарии при базе 5%, alpha=0.05, power=0.80:
5% -> 5.5%: 31,234 пользователей на группу
5% -> 6%: 8,158 пользователей на группу
5% -> 6.5%: 3,780 пользователей на группу


In [3]:
# Число НОВЫХ подходящих пользователей во всём эксперименте в день.
# Это пример, а не фактический трафик сайта.
eligible_users_per_day = 500
raw_recruitment_days = 2 * operational_arm / eligible_users_per_day
recruitment_days = max(14, ceil(raw_recruitment_days / 7) * 7)
followup_days = 14  # семь суток покупок, затем учёт возвратов до t0+14
print(f"Набор: {recruitment_days} дней")
print(f"Итого до закрытия окон последней когорты: {recruitment_days+followup_days} дней")


Набор: 35 дней
Итого до закрытия окон последней когорты: 49 дней


## Что нужно уточнить перед реальным запуском

1. Измерить конверсию по идентичной eligibility и зрелому семидневному окну. Подставить её вместо 5%.
2. Согласовать обнаруживаемый эффект и практическую значимость.
3. Отдельно рассчитать размер выборки для проверки неухудшения общей билетной выручки на пользователя. Нужны её среднее, вариативность, допустимая граница и предполагаемый эффект. Включать непокупателей с нулевой выручкой.
4. Выбрать максимум требуемых размеров для основной и критичной защитной метрик. Расчёт 8 158 на группу обеспечивает только указанную чувствительность конверсии.
5. Зафиксировать срок и метод анализа до просмотра результатов. Не продлевать тест ради значимости.

MDE +1 п.п. не является обещанным результатом и не превращается автоматически в порог наблюдённого успеха. Интервалы по выручке могут потребовать существенно большей выборки.
